# 📊 GEE Air Temperature Tile Downloader

This notebook efficiently downloads image tiles from the **Global Daily Air Temperature (2003-2020)** dataset for a user-specified region of interest.

## 🌍 Dataset Overview:
- **Coverage**: Global (50°S to 79°N) at 1km resolution
- **Time Period**: 2003-2020 daily data
- **Variables**: Maximum and minimum daily temperatures
- **Regional Collections**: 5 collections for efficient regional access

## ⚡ Key Features:
- **Automatic region detection** - Selects appropriate collection based on ROI
- **Efficient downloading** - Only clips to ROI, no full dataset downloads
- **Flexible date ranges** - Download specific years/periods
- **Multiple export options** - Individual images or time series stacks

In [1]:
# Import required libraries
import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import ipywidgets as widgets
from IPython.display import display, clear_output
import os
import time

# Set matplotlib to display inline
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

# Initialize Earth Engine with your project
try:
    ee.Initialize(project='tl-cities')
    print('✅ Earth Engine initialized successfully')
except Exception as e:
    print(f'❌ Earth Engine initialization failed: {e}')
    print('Please ensure you are authenticated and have the correct project ID')

print('📦 Available packages:')
print(f'   - geemap: {geemap.__version__}')
print(f'   - pandas: {pd.__version__}')
print(f'   - numpy: {np.__version__}')

# Create outputs directory
os.makedirs('../outputs', exist_ok=True)
print('📁 Created outputs directory: ../outputs')

/Users/martynclark/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


✅ Earth Engine initialized successfully
📦 Available packages:
   - geemap: 0.19.6
   - pandas: 1.4.4
   - numpy: 1.26.4
📁 Created outputs directory: ../outputs


## 🎯 Step 1: ROI Selection and Regional Collection Detection

In [2]:
# Global variables
analysis_geom = None
selected_collection = None
download_tasks = []

# Regional collection definitions
REGIONAL_COLLECTIONS = {
    'africa': {
        'id': 'projects/sat-io/open-datasets/global-daily-air-temp/africa',
        'name': 'Africa',
        'bounds': (-20, -40, 55, 40)  # west, south, east, north
    },
    'australia': {
        'id': 'projects/sat-io/open-datasets/global-daily-air-temp/australia', 
        'name': 'Australia',
        'bounds': (110, -50, 180, -5)
    },
    'europe_asia': {
        'id': 'projects/sat-io/open-datasets/global-daily-air-temp/europe_asia',
        'name': 'Europe & Asia',
        'bounds': (-15, 30, 180, 79)
    },
    'latin_america': {
        'id': 'projects/sat-io/open-datasets/global-daily-air-temp/latin_america',
        'name': 'Latin America', 
        'bounds': (-120, -60, -30, 35)
    },
    'north_america': {
        'id': 'projects/sat-io/open-datasets/global-daily-air-temp/north_america',
        'name': 'North America',
        'bounds': (-170, 15, -40, 79)
    }
}

def detect_regional_collection(geom):
    """Automatically detect which regional collection to use based on ROI centroid"""
    try:
        centroid = geom.centroid().coordinates().getInfo()
        lon, lat = centroid[0], centroid[1]
        
        print(f'🎯 ROI centroid: {lat:.3f}°N, {lon:.3f}°E')
        
        # Check which regional collection contains this point
        for region_key, region_info in REGIONAL_COLLECTIONS.items():
            west, south, east, north = region_info['bounds']
            
            if west <= lon <= east and south <= lat <= north:
                print(f'✅ Auto-selected: {region_info["name"]} collection')
                return region_key, region_info
        
        # Default fallback
        print('⚠️ No exact match found, defaulting to North America collection')
        return 'north_america', REGIONAL_COLLECTIONS['north_america']
        
    except Exception as e:
        print(f'❌ Error detecting region: {e}')
        return 'north_america', REGIONAL_COLLECTIONS['north_america']

def validate_collection_coverage(geom, collection_info):
    """Check if ROI is fully covered by the selected collection"""
    try:
        bounds = geom.bounds().getInfo()['coordinates'][0]
        roi_west, roi_south = bounds[0]
        roi_east, roi_north = bounds[2]
        
        col_west, col_south, col_east, col_north = collection_info['bounds']
        
        # Check if ROI is within collection bounds
        if (roi_west >= col_west and roi_east <= col_east and 
            roi_south >= col_south and roi_north <= col_north):
            print(f'✅ ROI fully covered by {collection_info["name"]} collection')
            return True
        else:
            print(f'⚠️ ROI may extend beyond {collection_info["name"]} collection bounds')
            print(f'   ROI: {roi_west:.1f} to {roi_east:.1f}°E, {roi_south:.1f} to {roi_north:.1f}°N')
            print(f'   Collection: {col_west:.1f} to {col_east:.1f}°E, {col_south:.1f} to {col_north:.1f}°N')
            return False
            
    except Exception as e:
        print(f'❌ Error validating coverage: {e}')
        return False

# Create map for ROI selection
m = geemap.Map(center=[0, 0], zoom=2)
m.add_basemap('SATELLITE')
m.add('draw_control')

def set_roi_from_drawing():
    """Extract ROI from map drawing and detect appropriate collection"""
    global analysis_geom, selected_collection
    
    try:
        if hasattr(m, 'draw_control') and len(m.draw_control.data) > 0:
            feature = m.draw_control.data[-1]
            coords = feature['geometry']['coordinates']
            
            if feature['geometry']['type'] == 'Polygon':
                analysis_geom = ee.Geometry.Polygon(coords)
            elif feature['geometry']['type'] == 'Rectangle':
                analysis_geom = ee.Geometry.Rectangle(coords)
            else:
                print('❌ Please draw a polygon or rectangle')
                return False
            
            # Calculate area
            area_km2 = analysis_geom.area().divide(1000000).getInfo()
            print(f'✅ ROI set: {area_km2:.1f} km²')
            
            # Auto-detect regional collection
            region_key, region_info = detect_regional_collection(analysis_geom)
            selected_collection = region_info
            
            # Validate coverage
            validate_collection_coverage(analysis_geom, region_info)
            
            # Update collection selector widget
            collection_selector.value = region_key
            
            return True
        else:
            print('❌ No drawing found. Please draw a polygon or rectangle on the map.')
            return False
    except Exception as e:
        print(f'❌ Error setting ROI: {e}')
        return False

def set_roi_from_coordinates():
    """Set ROI from coordinate inputs"""
    global analysis_geom, selected_collection
    
    try:
        west = float(west_input.value)
        east = float(east_input.value) 
        south = float(south_input.value)
        north = float(north_input.value)
        
        if west >= east or south >= north:
            print('❌ Invalid coordinates: west < east and south < north')
            return False
        
        analysis_geom = ee.Geometry.Rectangle([west, south, east, north])
        area_km2 = analysis_geom.area().divide(1000000).getInfo()
        
        # Visualize ROI on map
        roi_image = ee.Image().paint(analysis_geom, 1, 2)
        m.addLayer(roi_image, {'palette': ['red'], 'max': 1}, 'ROI')
        m.centerObject(analysis_geom)
        
        print(f'✅ ROI set: {area_km2:.1f} km²')
        print(f'   Bounds: {west:.3f}°W to {east:.3f}°E, {south:.3f}°S to {north:.3f}°N')
        
        # Auto-detect regional collection
        region_key, region_info = detect_regional_collection(analysis_geom)
        selected_collection = region_info
        
        # Validate coverage
        validate_collection_coverage(analysis_geom, region_info)
        
        # Update collection selector widget
        collection_selector.value = region_key
        
        return True
    except Exception as e:
        print(f'❌ Error setting ROI: {e}')
        return False

def on_collection_change(change):
    """Handle manual collection selection change"""
    global selected_collection
    selected_collection = REGIONAL_COLLECTIONS[change['new']]
    print(f'📡 Manually selected: {selected_collection["name"]} collection')
    
    if analysis_geom is not None:
        validate_collection_coverage(analysis_geom, selected_collection)

# ROI input widgets
west_input = widgets.FloatText(value=-120.0, description='West (°):')
east_input = widgets.FloatText(value=-80.0, description='East (°):')
south_input = widgets.FloatText(value=25.0, description='South (°):')
north_input = widgets.FloatText(value=50.0, description='North (°):')

# Collection selector (manual override)
collection_selector = widgets.Dropdown(
    options=[(info['name'], key) for key, info in REGIONAL_COLLECTIONS.items()],
    value='north_america',
    description='Collection:'
)
collection_selector.observe(on_collection_change, names='value')

# Action buttons
set_drawing_button = widgets.Button(description='📍 Use Drawing', button_style='success')
set_coords_button = widgets.Button(description='📍 Use Coordinates', button_style='info')

set_drawing_button.on_click(lambda b: set_roi_from_drawing())
set_coords_button.on_click(lambda b: set_roi_from_coordinates())

roi_interface = widgets.VBox([
    widgets.HTML('<h3>🎯 ROI Selection & Collection Detection</h3>'),
    widgets.HTML('<b>Method 1: Draw on Map</b>'),
    set_drawing_button,
    widgets.HTML('<b>Method 2: Enter Coordinates</b>'),
    widgets.HBox([west_input, east_input]),
    widgets.HBox([south_input, north_input]),
    set_coords_button,
    widgets.HTML('<b>Regional Collection (auto-detected, can override):</b>'),
    collection_selector
])

display(roi_interface)
display(m)

print('🎯 ROI Selection Ready')
print('✨ Collection will be auto-detected based on ROI location')

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=HBox(children=(Toggl…

🎯 ROI Selection Ready
✨ Collection will be auto-detected based on ROI location


## 📅 Step 2: Download Configuration

In [3]:
# Download configuration widgets
start_year = widgets.IntSlider(value=2020, min=2003, max=2020, description='Start Year:')
end_year = widgets.IntSlider(value=2020, min=2003, max=2020, description='End Year:')

# Temperature variable selection
temp_variable = widgets.Dropdown(
    options=[('Maximum Temperature', 'tmax'), ('Minimum Temperature', 'tmin'), ('Both', 'both')],
    value='tmax',
    description='Variable:'
)

# Download mode selection
download_mode = widgets.Dropdown(
    options=[
        ('Individual Images', 'individual'),
        ('Monthly Stacks', 'monthly'),
        ('Annual Stacks', 'annual'),
        ('Complete Time Series', 'timeseries')
    ],
    value='monthly',
    description='Download Mode:'
)

# Export resolution
export_scale = widgets.IntSlider(
    value=1000,
    min=500,
    max=5000,
    step=500,
    description='Resolution (m):'
)

# Export format
export_format = widgets.Dropdown(
    options=[('GeoTIFF', 'GeoTIFF'), ('NetCDF', 'NetCDF')],
    value='GeoTIFF',
    description='Format:'
)

# Export folder
export_folder = widgets.Text(
    value='air_temp_downloads',
    description='GDrive Folder:',
    placeholder='Enter Google Drive folder name'
)

config_interface = widgets.VBox([
    widgets.HTML('<h3>📅 Download Configuration</h3>'),
    widgets.HTML('<div style="background-color: #e8f4fd; padding: 10px; border-radius: 5px;">'
                '<b>Efficient Download Strategy:</b><br>'
                '• Only processes ROI extent (no full dataset downloads)<br>'
                '• Multiple download modes for different use cases<br>'
                '• Automatic task management and progress tracking</div>'),
    widgets.HBox([start_year, end_year]),
    widgets.HBox([temp_variable, download_mode]),
    widgets.HBox([export_scale, export_format]),
    export_folder
])

display(config_interface)
print('📅 Download Configuration Ready')

📅 Download Configuration Ready


## 📥 Step 3: Image Download and Task Management

In [4]:
def get_collection_images(collection_id, start_date, end_date, temp_type, geometry):
    """Get filtered image collection for download"""
    collection = ee.ImageCollection(collection_id)
    
    # Filter collection
    filtered = (collection
                .filterDate(start_date, end_date)
                .filterBounds(geometry)
                .filter(ee.Filter.eq('prop_type', temp_type)))
    
    # Process images: scale to Celsius, clip to ROI, add date info
    def process_image(img):
        processed = (img.select('b1')
                    .divide(10)  # Scale to Celsius
                    .rename(f'temp_{temp_type}')
                    .clip(geometry)
                    .copyProperties(img, ['system:time_start', 'prop_type']))
        
        # Add date band for easier processing
        date = ee.Date(img.get('system:time_start'))
        return processed.set({
            'date': date.format('YYYY-MM-dd'),
            'year': date.get('year'),
            'month': date.get('month'),
            'day': date.get('day')
        })
    
    return filtered.map(process_image)

def create_download_tasks():
    """Create download tasks based on configuration"""
    global download_tasks
    
    if analysis_geom is None:
        print('❌ Please set an ROI first!')
        return False
    
    if selected_collection is None:
        print('❌ No collection selected!')
        return False
    
    try:
        print('🔄 Creating download tasks...')
        
        # Get configuration
        start_yr = start_year.value
        end_yr = end_year.value
        var_type = temp_variable.value
        dl_mode = download_mode.value
        scale = export_scale.value
        fmt = export_format.value
        folder = export_folder.value
        
        collection_id = selected_collection['id']
        
        download_tasks = []
        variables_to_process = ['tmax', 'tmin'] if var_type == 'both' else [var_type]
        
        for var in variables_to_process:
            print(f'   📊 Processing variable: {var}')
            
            if dl_mode == 'individual':
                # Individual daily images
                for year in range(start_yr, end_yr + 1):
                    collection = get_collection_images(
                        collection_id, f'{year}-01-01', f'{year}-12-31', var, analysis_geom
                    )
                    
                    image_list = collection.getInfo()['features']
                    print(f'      📅 {year}: {len(image_list)} images')
                    
                    for img_info in image_list:
                        date_str = img_info['properties']['date']
                        img_id = img_info['id']
                        
                        task_config = {
                            'image': ee.Image(img_id),
                            'description': f'airtemp_{var}_{date_str}',
                            'folder': folder,
                            'region': analysis_geom,
                            'scale': scale,
                            'fileFormat': fmt,
                            'maxPixels': 1e9
                        }
                        
                        download_tasks.append(('export', task_config))
            
            elif dl_mode == 'monthly':
                # Monthly composite images
                for year in range(start_yr, end_yr + 1):
                    for month in range(1, 13):
                        start_date = f'{year}-{month:02d}-01'
                        if month == 12:
                            end_date = f'{year + 1}-01-01'
                        else:
                            end_date = f'{year}-{month + 1:02d}-01'
                        
                        collection = get_collection_images(
                            collection_id, start_date, end_date, var, analysis_geom
                        )
                        
                        if collection.size().getInfo() > 0:
                            # Create monthly mean
                            monthly_mean = collection.mean().set({
                                'year': year,
                                'month': month,
                                'variable': var
                            })
                            
                            task_config = {
                                'image': monthly_mean,
                                'description': f'airtemp_{var}_{year}_{month:02d}_monthly',
                                'folder': folder,
                                'region': analysis_geom,
                                'scale': scale,
                                'fileFormat': fmt,
                                'maxPixels': 1e9
                            }
                            
                            download_tasks.append(('export', task_config))
            
            elif dl_mode == 'annual':
                # Annual composite images
                for year in range(start_yr, end_yr + 1):
                    collection = get_collection_images(
                        collection_id, f'{year}-01-01', f'{year}-12-31', var, analysis_geom
                    )
                    
                    if collection.size().getInfo() > 0:
                        # Create annual statistics
                        annual_mean = collection.mean().rename(f'{var}_mean')
                        annual_max = collection.max().rename(f'{var}_max')
                        annual_min = collection.min().rename(f'{var}_min')
                        
                        annual_stats = annual_mean.addBands([annual_max, annual_min]).set({
                            'year': year,
                            'variable': var
                        })
                        
                        task_config = {
                            'image': annual_stats,
                            'description': f'airtemp_{var}_{year}_annual_stats',
                            'folder': folder,
                            'region': analysis_geom,
                            'scale': scale,
                            'fileFormat': fmt,
                            'maxPixels': 1e9
                        }
                        
                        download_tasks.append(('export', task_config))
            
            elif dl_mode == 'timeseries':
                # Complete time series as image collection
                collection = get_collection_images(
                    collection_id, f'{start_yr}-01-01', f'{end_yr}-12-31', var, analysis_geom
                )
                
                total_images = collection.size().getInfo()
                print(f'      📊 Total images in time series: {total_images}')
                
                if total_images > 0:
                    # Export as a collection (requires special handling)
                    print(f'      ⚠️ Time series mode: Will create individual tasks for each image')
                    
                    image_list = collection.getInfo()['features']
                    for img_info in image_list:
                        date_str = img_info['properties']['date']
                        img_id = img_info['id']
                        
                        task_config = {
                            'image': ee.Image(img_id),
                            'description': f'timeseries_{var}_{date_str}',
                            'folder': folder,
                            'region': analysis_geom,
                            'scale': scale,
                            'fileFormat': fmt,
                            'maxPixels': 1e9
                        }
                        
                        download_tasks.append(('export', task_config))
        
        print(f'\n✅ Created {len(download_tasks)} download tasks')
        
        if len(download_tasks) > 100:
            print('⚠️ Large number of tasks - consider using monthly or annual mode for efficiency')
        
        return True
        
    except Exception as e:
        print(f'❌ Error creating tasks: {e}')
        import traceback
        print(f'   Details: {traceback.format_exc()}')
        return False

def start_downloads():
    """Start all download tasks"""
    global download_tasks
    
    if not download_tasks:
        print('❌ No download tasks created. Please create tasks first!')
        return False
    
    try:
        print(f'🚀 Starting {len(download_tasks)} download tasks...')
        print('📁 Files will be saved to Google Drive in folder:', export_folder.value)
        
        active_tasks = []
        
        for i, (task_type, config) in enumerate(download_tasks):
            if task_type == 'export':
                try:
                    # Start export task
                    task = ee.batch.Export.image.toDrive(**config)
                    task.start()
                    
                    active_tasks.append({
                        'task': task,
                        'description': config['description'],
                        'started': time.time()
                    })
                    
                    print(f'   ✅ Started task {i+1}/{len(download_tasks)}: {config["description"]}')
                    
                    # Add small delay to avoid overwhelming the API
                    if i % 10 == 0 and i > 0:
                        time.sleep(1)
                        
                except Exception as e:
                    print(f'   ❌ Failed to start task {i+1}: {e}')
                    continue
        
        print(f'\n🎉 Successfully started {len(active_tasks)} download tasks!')
        print('\n📊 Task Status Summary:')
        print(f'   - Tasks started: {len(active_tasks)}')
        print(f'   - Export format: {export_format.value}')
        print(f'   - Resolution: {export_scale.value}m')
        print(f'   - Google Drive folder: {export_folder.value}')
        
        print('\n⏳ Downloads are now running in Google Earth Engine.')
        print('   Check your Google Drive folder and GEE Code Editor Tasks tab for progress.')
        print('   Use the task monitoring function below to check status.')
        
        return active_tasks
        
    except Exception as e:
        print(f'❌ Error starting downloads: {e}')
        import traceback
        print(f'   Details: {traceback.format_exc()}')
        return False

def check_task_status():
    """Check status of running tasks"""
    try:
        print('📊 Checking task status...')
        
        tasks = ee.batch.Task.list()
        
        # Filter for our tasks
        folder_name = export_folder.value
        our_tasks = [task for task in tasks 
                    if 'airtemp' in task.config.get('description', '') or 
                       'timeseries' in task.config.get('description', '')]
        
        if not our_tasks:
            print('❌ No matching tasks found')
            return
        
        # Count by status
        status_counts = {}
        for task in our_tasks[:20]:  # Show first 20 tasks
            status = task.state
            status_counts[status] = status_counts.get(status, 0) + 1
            
            print(f'   {task.config["description"]}: {status}')
        
        print(f'\n📈 Status Summary (showing first {min(20, len(our_tasks))} of {len(our_tasks)} tasks):')
        for status, count in status_counts.items():
            print(f'   {status}: {count}')
        
    except Exception as e:
        print(f'❌ Error checking status: {e}')

# Download control buttons
create_tasks_button = widgets.Button(description='🔄 Create Download Tasks', button_style='primary')
start_downloads_button = widgets.Button(description='🚀 Start Downloads', button_style='success')
check_status_button = widgets.Button(description='📊 Check Status', button_style='info')

create_tasks_button.on_click(lambda b: create_download_tasks())
start_downloads_button.on_click(lambda b: start_downloads())
check_status_button.on_click(lambda b: check_task_status())

download_interface = widgets.VBox([
    widgets.HTML('<h3>📥 Download Management</h3>'),
    widgets.HTML('<div style="background-color: #fff3cd; padding: 10px; border-radius: 5px;">'
                '<b>Download Process:</b><br>'
                '1. <b>Create Tasks:</b> Generate download jobs based on your configuration<br>'
                '2. <b>Start Downloads:</b> Submit tasks to Google Earth Engine<br>'
                '3. <b>Check Status:</b> Monitor progress and completion<br>'
                '📁 <b>Files save to:</b> Google Drive → {folder_name}</div>'.format(folder_name=export_folder.value)),
    widgets.HBox([create_tasks_button, start_downloads_button, check_status_button])
])

display(download_interface)
print('📥 Download Management Ready')
print('💡 Tip: Start with a small test (1 year, monthly mode) before large downloads')

📥 Download Management Ready
💡 Tip: Start with a small test (1 year, monthly mode) before large downloads


## 📊 Step 4: Data Preview and Validation

In [5]:
def preview_data():
    """Preview a sample image to validate the setup"""
    if analysis_geom is None:
        print('❌ Please set an ROI first!')
        return
    
    if selected_collection is None:
        print('❌ No collection selected!')
        return
    
    try:
        print('🔍 Loading sample image for preview...')
        
        collection_id = selected_collection['id']
        var_type = temp_variable.value if temp_variable.value != 'both' else 'tmax'
        
        # Get a recent image for preview
        sample_collection = get_collection_images(
            collection_id, '2020-07-01', '2020-07-31', var_type, analysis_geom
        )
        
        count = sample_collection.size().getInfo()
        if count == 0:
            print('❌ No images found for preview. Check ROI and collection coverage.')
            return
        
        print(f'   📊 Found {count} images in July 2020')
        
        # Get first image
        sample_image = sample_collection.first()
        
        # Get image info
        image_info = sample_image.getInfo()
        print(f'   📅 Sample image: {image_info["properties"].get("date", "Unknown date")}')
        
        # Add to map for visualization
        vis_params = {
            'min': 10,
            'max': 40,
            'palette': ['blue', 'cyan', 'yellow', 'orange', 'red']
        }
        
        m.addLayer(sample_image, vis_params, f'Sample {var_type.upper()} (°C)')
        m.centerObject(analysis_geom)
        
        # Get some statistics
        stats = sample_image.reduceRegion(
            reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True),
            geometry=analysis_geom,
            scale=1000,
            maxPixels=1e9
        ).getInfo()
        
        band_name = f'temp_{var_type}'
        if stats:
            min_temp = stats.get(f'{band_name}_min', 'N/A')
            max_temp = stats.get(f'{band_name}_max', 'N/A')
            mean_temp = stats.get(f'{band_name}_mean', 'N/A')
            
            print(f'\n📊 Sample Image Statistics:')
            print(f'   Min temperature: {min_temp:.1f}°C' if isinstance(min_temp, (int, float)) else f'   Min temperature: {min_temp}')
            print(f'   Max temperature: {max_temp:.1f}°C' if isinstance(max_temp, (int, float)) else f'   Max temperature: {max_temp}')
            print(f'   Mean temperature: {mean_temp:.1f}°C' if isinstance(mean_temp, (int, float)) else f'   Mean temperature: {mean_temp}')
        
        print('\n✅ Preview complete! Check the map for visualization.')
        print('🎯 If the preview looks correct, proceed with download creation.')
        
    except Exception as e:
        print(f'❌ Error creating preview: {e}')
        import traceback
        print(f'   Details: {traceback.format_exc()}')

def validate_setup():
    """Validate the current setup before downloads"""
    print('🔍 Validating setup...')
    
    issues = []
    
    # Check ROI
    if analysis_geom is None:
        issues.append('❌ No ROI defined')
    else:
        area_km2 = analysis_geom.area().divide(1000000).getInfo()
        if area_km2 > 100000:  # Very large area
            issues.append(f'⚠️ Large ROI ({area_km2:.0f} km²) - downloads may take very long')
        else:
            print(f'   ✅ ROI size: {area_km2:.1f} km²')
    
    # Check collection
    if selected_collection is None:
        issues.append('❌ No collection selected')
    else:
        print(f'   ✅ Collection: {selected_collection["name"]}')
    
    # Check date range
    start_yr = start_year.value
    end_yr = end_year.value
    
    if start_yr > end_yr:
        issues.append('❌ Start year > End year')
    elif end_yr - start_yr > 5:
        issues.append(f'⚠️ Long time period ({end_yr - start_yr + 1} years) - many download tasks')
    else:
        print(f'   ✅ Date range: {start_yr}-{end_yr}')
    
    # Check download mode vs time period
    dl_mode = download_mode.value
    total_years = end_yr - start_yr + 1
    
    if dl_mode == 'individual':
        estimated_tasks = total_years * 365  # Rough estimate
        if estimated_tasks > 1000:
            issues.append(f'⚠️ Individual mode will create ~{estimated_tasks} tasks - consider monthly/annual')
    elif dl_mode == 'monthly':
        estimated_tasks = total_years * 12
        print(f'   ✅ Estimated tasks: ~{estimated_tasks} (monthly mode)')
    elif dl_mode == 'annual':
        print(f'   ✅ Estimated tasks: {total_years} (annual mode)')
    
    # Check export settings
    if not export_folder.value.strip():
        issues.append('❌ No export folder specified')
    else:
        print(f'   ✅ Export folder: {export_folder.value}')
    
    print(f'   ✅ Resolution: {export_scale.value}m')
    print(f'   ✅ Format: {export_format.value}')
    print(f'   ✅ Variable: {temp_variable.value}')
    
    # Summary
    if issues:
        print('\n⚠️ Issues found:')
        for issue in issues:
            print(f'   {issue}')
        print('\n🔧 Please address these issues before proceeding.')
        return False
    else:
        print('\n✅ Setup validation passed!')
        print('🚀 Ready to create download tasks.')
        return True

# Preview and validation buttons
preview_button = widgets.Button(description='🔍 Preview Data', button_style='info')
validate_button = widgets.Button(description='✅ Validate Setup', button_style='warning')

preview_button.on_click(lambda b: preview_data())
validate_button.on_click(lambda b: validate_setup())

validation_interface = widgets.VBox([
    widgets.HTML('<h3>📊 Data Preview & Validation</h3>'),
    widgets.HTML('<div style="background-color: #d4edda; padding: 10px; border-radius: 5px;">'
                '<b>Pre-Download Checks:</b><br>'
                '• <b>Preview Data:</b> Visualize a sample image to verify setup<br>'
                '• <b>Validate Setup:</b> Check configuration for potential issues</div>'),
    widgets.HBox([preview_button, validate_button])
])

display(validation_interface)
print('📊 Preview & Validation Ready')

📊 Preview & Validation Ready


## 📋 Summary

This notebook provides an **efficient tile downloader** for the Global Daily Air Temperature dataset:

### 🌍 **Dataset Features:**
- **Coverage**: Global (50°S to 79°N) at 1km resolution
- **Time Period**: 2003-2020 daily data
- **Variables**: Maximum and minimum daily temperatures
- **Regional Collections**: 5 optimized collections for efficient access

### ⚡ **Efficiency Features:**
- **Automatic region detection** - Selects optimal collection based on ROI
- **Multiple download modes** - Individual, monthly, annual, or complete time series
- **ROI-only processing** - No unnecessary data downloads
- **Task management** - Automated batch processing with progress monitoring

### 🚀 **Workflow:**
1. **Set ROI** - Draw on map or enter coordinates
2. **Configure Download** - Choose variables, time period, and output format
3. **Preview & Validate** - Check setup before large downloads
4. **Create & Start Tasks** - Automated batch download to Google Drive

### 💡 **Best Practices:**
- Start with **monthly mode** for large time periods
- Use **annual mode** for multi-year statistical analysis
- **Preview data** first to verify coverage and quality
- Monitor tasks in **GEE Code Editor** for progress tracking

**Perfect for**: Regional climate analysis, heat studies, temperature trend analysis, and any application requiring efficient access to high-resolution daily temperature data! 🌡️